# Домашнее задание №20: Ансамбли моделей машинного обучения. Ансамбль стекинга. (Практика)

In [4]:
import pandas as pd
import numpy as np
import optuna
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.model_selection import cross_val_predict, train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import make_pipeline

# Загрузка данных
data = pd.read_csv('./data/SouthGermanCredit_encoded.csv')
raw_X = data.iloc[:, :-1]
y = data.iloc[:, -1]

X_train, X_test, y_train, y_test = train_test_split(
    raw_X, y, test_size=0.2, random_state=42, stratify=y)

# Objective для Optuna: подбираем alpha для Lasso и Ridge
def objective(trial):
    alpha_lasso = trial.suggest_float('alpha_lasso', 1e-4, 1e2, log=True)
    alpha_ridge = trial.suggest_float('alpha_ridge', 1e-2, 1e3, log=True)

    # OOF-предсказания (out-of-fold) для стекинга — чтобы не было утечки
    # Каждая модель обучается на 4 фолдах и предсказывает 5-й, потом меняются
    lr_pred = cross_val_predict(
        make_pipeline(MinMaxScaler(), LinearRegression()),
        X_train, y_train, cv=5)

    lasso_pred = cross_val_predict(
        make_pipeline(MinMaxScaler(), Lasso(alpha=alpha_lasso, max_iter=10000)),
        X_train, y_train, cv=5)

    ridge_pred = cross_val_predict(
        make_pipeline(MinMaxScaler(), Ridge(alpha=alpha_ridge)),
        X_train, y_train, cv=5)

    # Ансамбль: усреднение предсказаний (простейший стекинг)
    ensemble_pred = (lr_pred + lasso_pred + ridge_pred) / 3
    ensemble_class = (ensemble_pred > 0.5).astype(int)

    return accuracy_score(y_train, ensemble_class)

# Запуск Optuna
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=500)

print('Лучшие альфы:', study.best_params)
print('Точность ансамбля на train (CV):', study.best_value)

# Финальное обучение с лучшими альфами
best_alpha_lasso = study.best_params['alpha_lasso']
best_alpha_ridge = study.best_params['alpha_ridge']

lr_final = make_pipeline(MinMaxScaler(), LinearRegression()).fit(X_train, y_train)
lasso_final = make_pipeline(MinMaxScaler(), Lasso(alpha=best_alpha_lasso, max_iter=10000)).fit(X_train, y_train)
ridge_final = make_pipeline(MinMaxScaler(), Ridge(alpha=best_alpha_ridge)).fit(X_train, y_train)

# Предсказания на тесте
lr_test = lr_final.predict(X_test)
lasso_test = lasso_final.predict(X_test)
ridge_test = ridge_final.predict(X_test)

# Ансамбль на тесте
ensemble_test = (lr_test + lasso_test + ridge_test) / 3
ensemble_class = (ensemble_test > 0.5).astype(int)

# Сравнение
print('\nТочность на тесте:')
print(f'LinearRegression: {accuracy_score(y_test, (lr_test > 0.5).astype(int)):.4f}')
print(f'Lasso (alpha={best_alpha_lasso:.4f}): {accuracy_score(y_test, (lasso_test > 0.5).astype(int)):.4f}')
print(f'Ridge (alpha={best_alpha_ridge:.4f}): {accuracy_score(y_test, (ridge_test > 0.5).astype(int)):.4f}')
print(f'Ансамбль (среднее): {accuracy_score(y_test, ensemble_class):.4f}')

print('\nМатрица ошибок ансамбля:')
print(confusion_matrix(y_test, ensemble_class))

Лучшие альфы: {'alpha_lasso': 0.00015725745249590674, 'alpha_ridge': 92.92087790838995}
Точность ансамбля на train (CV): 0.75125

Точность на тесте:
LinearRegression: 0.7650
Lasso (alpha=0.0002): 0.8050
Ridge (alpha=92.9209): 0.7550
Ансамбль (среднее): 0.7850

Матрица ошибок ансамбля:
[[ 32  28]
 [ 15 125]]


- Хороших ловит отлично: 125 из 140 (~89%).
- Плохих ловит плохо: 32 из 40 (~53%), 28 плохих прошли как хорошие